<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test For Orbit

Difference with FiS and related

In [ ]:
import os

# Test
# Set this BEFORE importing pytorch/tensorflow
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
# Check if it worked (should return 1 if you selected a single GPU)
print(torch.cuda.device_count()) 

In [ ]:
# DS_ID = "NONHUMAN-RESEARCH/general-task-index"
DS_ID = "NONHUMAN-RESEARCH/pick-and-place-all-fruits-annotated"
PRETRAINED_PATH = "lerobot/pi05_base"

In [ ]:
from xhuman.policies.orbit.configuration_orbit import OrbitConfig

policy_config = OrbitConfig(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

In [ ]:
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [ ]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [ ]:
from xhuman.datasets.factory import make_dataset_xhuman

dataset = make_dataset_xhuman(cfg)

In [ ]:
from xhuman.policies.factory import make_xhuman_policy

policy = make_xhuman_policy(
    cfg=cfg.policy,
    ds_meta=dataset.meta,
)

In [ ]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
        dataset_stats=dataset.meta.stats,
    )

In [ ]:
device = torch.device("cuda")

In [ ]:
frame = dataset[0]
frame.keys()

In [ ]:
batch_slow = preprocessor.slow(frame)

In [ ]:
batch_fast = preprocessor.fast(frame)

In [ ]:
batch_fast["observation.state"]

In [ ]:
batch_slow.keys()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

In [ ]:
tokens_pt = batch_slow["observation.language.tokens"]
words = tokenizer.batch_decode(tokens_pt, skip_special_tokens=True)
words

In [ ]:
prefix_out , kv_cache , pad_mask  = policy.sample_embedding(batch_slow)

In [ ]:
prefix_out[0].shape

In [ ]:
batch_slow.keys()

In [ ]:
img_fast, img_masks_fast = policy._preprocess_images(batch_fast)
raw_state = batch_fast["observation.state"]
raw_state

In [ ]:
batch_slow["observation.state"]

In [ ]:
policy.model.config.output_features["action"].shape

In [ ]:
actions = policy.model.sample_actions(
    images=img_fast,
    img_masks=img_masks_fast,
    tokens=tokens_slow,
    masks=masks_slow,
)
actions.shape

In [ ]:
actions = policy.model.fast_model(
    images_fast=img_fast,
    img_masks_fast=img_masks_fast,
    past_key_values=kv_cache,
    state_embs=raw_state,
    latent_embeds=prefix_out[0],
    prefix_pad_masks=pad_mask,
)
actions.shape